In [1]:
#Task 4
# Aynı modeli Türkçe metinle eğit. Metni kendin derle, Tiny Shakespeare'e
# yakın boyutta olsun (yaklaşık 1 MB). Kaynaklardaki Vikikaynak'tan
# başlayabilirsin. Ürettiği metinden örnek ver. Türkçe metnin kaç farklı
# karakteri var, Shakespeare'inkiyle karşılaştır. İki val loss'u doğrudan
# karşılaştırabilir misin, neden, yaz.

In [2]:
device = 'cuda'
dropout = 0.2
n_head = 6
n_layer = 6
import time
import torch
import torch.nn as nn
from torch.nn import functional as F
with open("altayli_kose_yazilari.txt", "r", encoding = "utf-8") as f:
    text = f.read()

In [3]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(vocab_size)

103


In [4]:
#mapping
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}

In [5]:
#decoder & encoder
def encode(s):
    integers = [stoi[i] for i in s]
    return integers

def decode(l):
    lst = "".join([itos[i] for i in l])
    return lst

In [6]:
data = torch.tensor(encode(text), dtype=torch.long)

In [7]:
#train val split with 90% 10%
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [8]:
#the way pick our batches
torch.manual_seed(1337)
batch_size = 4
block_size = 256

def get_batch(split):

    data = train_data if split == "train" else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,)) #do not want to grab latest datas like last data point as a starter
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x,y

In [9]:
#self attention head as a class
class Head(nn.Module):

    def __init__(self, head_size):
        super().__init__()
        self.head_size = head_size
        self.key = nn.Linear(emb_dim, head_size, bias = False)
        self.query = nn.Linear(emb_dim, head_size, bias = False)
        self.value = nn.Linear(emb_dim, head_size, bias = False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size,block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self,x):
        B,T,C =x.shape
        k = self.key(x)
        q = self.query(x)
        v = self.value(x)

        wei = q @ k.transpose(-2,-1) * self.head_size**-0.5
        wei = wei.masked_fill(self.tril[:T,:T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)

        out = wei @ v
        return out

In [10]:
#multi head self attention
class MultiHeadAttention(nn.Module):

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(emb_dim, emb_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim = -1)
        out = self.dropout(self.proj(out))
        return out


In [11]:
#simple mlp with ReLU
class FeedForward(nn.Module):

    def __init__(self, emb_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(emb_dim,4 * emb_dim),
            nn.ReLU(),
            nn.Linear(4 * emb_dim, emb_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):

    def __init__(self,emb_dim, n_head):
        super().__init__()
        head_size = emb_dim // n_head
        self.slfattention_head = MultiHeadAttention(n_head, head_size)
        self.fddfrwd = FeedForward(emb_dim)
        self.ln1 = nn.LayerNorm(emb_dim)
        self.ln2 = nn.LayerNorm(emb_dim)

    def forward(self,x):
        x= x + self.slfattention_head(self.ln1(x))
        x= x + self.fddfrwd(self.ln2(x))
        return x

In [12]:
#model as a class
emb_dim = 384
class BigramLanguageModel(nn.Module):

    def __init__(self,vocab_size):
        super().__init__()

        self.token_embedding_table = nn.Embedding(vocab_size,emb_dim)   #token embeddings
        self.positional_embedding_table = nn.Embedding(block_size,emb_dim) #position embeddings
       ##adding our blocks
        self.blocks = nn.Sequential(*[Block(emb_dim, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(emb_dim)
        self.lm_head = nn.Linear(emb_dim, vocab_size)

    def forward(self, idx, targets=None):
        B,T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.positional_embedding_table(torch.arange(T,device=device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            loss =None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T,C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self,idx,max_new_tokens):

        for _ in range(max_new_tokens):

            idx_cond = idx[:,-block_size:]

            logits, loss =self(idx_cond)

            logits = logits[:,-1,:]

            probs = F.softmax(logits,dim=-1)

            idx_next = torch.multinomial(probs,num_samples=1)

            idx = torch.cat((idx,idx_next), dim=1)
        return idx

m = BigramLanguageModel(vocab_size).to(device)
sum(p.numel() for p in m.parameters())

10818151

In [13]:
#lets train our model
batch_size = 64
epoch = 5000

startt=time.time()
optimizer = torch.optim.AdamW(m.parameters(), lr=3e-4)

for _ in range(epoch):

    xb,yb = get_batch("train") #first gather batch
    logits, loss = m(xb,yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(f'the train loss is {loss.item()}')
end=time.time()
print(f'Training took {(end-startt)/60} minute')

the train loss is 1.1028203964233398
Training took 9.364605180422465 minute


In [14]:
#val loss estimator
m.eval()
@torch.no_grad()
def val_loss_estimator():
    losses = torch.zeros(200)
    for _ in range (200):
        x,y = get_batch("val")
        logits, loss = m(x,y)
        losses[_]= loss.item()
    print(f'the validation loss is around {losses.mean()}')
val_loss_estimator()

the validation loss is around 1.3117446899414062


In [15]:
start = torch.tensor([encode("NE ZAMAN ADAM OLURUZ?")], dtype=torch.long, device=device)
out = m.generate(start, 2000)
print(decode(out[0].tolist()))

NE ZAMAN ADAM OLURUZ?
Bir süresinde anımsal olmadığı zaman

Müslümanlık nuresinin sevdiği’den geçmediği gibi belli ki, bizim de yurt dışında haksızlık reklam vermiyorum ama bu adam,

Bir de ‘ABV’de en yüksek kapıda tartışma, Piyasa da bunlar güçsün değiştirmişti,

Sonra Türkiye’nin “Biz metrekarel” çözümünü ahlakkıyla değil de, kullanmadığını biliyordu,

Lütfen Suda iddia ediyorsun, kendisine çürüt reklam veriyorsun, yani vermediğini iktidar yalan yine evleneliğini düşündüğü ve türbanını benim görevimi yaynından kıyacaksınız,

Hürriyet ne yanıt vermemiş,

Bir kaç yıl önce Umur Talu Talu’ya yine vermemişler,

Özgürlüğü, ya da bu kadar ederken daha da böyle görev böyle bir demokrasi olsa bile her yerde yok,

Başbakan’ın Yardımcısı ve Lexus RX RX internet sitesine bile dahi ve farkı o dönemde bir Chelsea’yi gezmek, 300 Ahmet Çalık’ta yine bir para aldık,

Ahmet Çalık’tan haberini Hora’nın uçuşa gitti,

Manşet Sürekli İslam kapatma davası Yalçın, Ancak Hollanda, Hayrunissa Hanım Hanım’ın m